### <div style="background-color:blue; color:white; padding:10px;"> Imports </div>

In [8]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

### <div style="background-color:blue; color:white; padding:10px;"> Paths and parameters </div>

In [9]:
# Dataset paths
RGB_ROOT = r"D:\Datasets\Datasets\EPIC_Kitchen\RGB\P01_05"
FLOW_ROOT = r"D:\Datasets\Datasets\EPIC_Kitchen\OpticalFlow\P01_05\P01_05"
ANNOTATION_CSV = r"D:\Datasets\Datasets\EPIC_Kitchen\Label\P01_05.csv"
# Output paths
OUTPUT_FEATURES_CSV = "../Features/P01_05_fused_features.csv"
OUTPUT_LABELS_CSV = "../Labels/P01_05_labels.csv"
# Model parameters
FEATURE_DIM = 512
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


### <div style="background-color:blue; color:white; padding:10px;">Load annotation CSV</div>

In [10]:
annotations = pd.read_csv(ANNOTATION_CSV)
print("Total annotations:", len(annotations))
annotations.head()

Total annotations: 259


,StartFrame,EndFrame,Verb,Verb_class,Noun,Noun_class,ActionLabel,ActionName
0,248,355,open,2,fridge,10,0,open fridge
1,390,484,take,0,mushroom,110,1,take mushroom
2,481,522,move,9,container,29,2,move container
3,524,853,take,0,sausage,84,3,take sausage
4,849,973,put,1,mushroom,110,4,put mushroom


### <div style="background-color:blue; color:white; padding:10px;"> Create frame-level multi-label matrix </div>

In [11]:
max_frame = annotations['EndFrame'].max()
num_classes = annotations['ActionLabel'].nunique()
print("Total frames:", max_frame)
print("Total classes:", num_classes)

Total frames: 76139
Total classes: 123


In [12]:
labels = np.zeros((max_frame + 1, num_classes), dtype=np.float32)
for _, row in annotations.iterrows():    
    start = int(row['StartFrame'])
    end = int(row['EndFrame'])
    cls = int(row['ActionLabel'])    
    labels[start:end+1, cls] = 1.0
print("Labels shape:", labels.shape)

Labels shape: (76140, 123)


### <div style="background-color:blue; color:white; padding:10px;"> Define image transform </div>

In [13]:
transform = transforms.Compose([    
    transforms.Resize((224, 224)),    
    transforms.ToTensor(),    
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

### <div style="background-color:blue; color:white; padding:10px;">Load ResNet-50 feature extractor </div>

In [14]:
resnet = models.resnet50(weights=True)
# Remove classifier
modules = list(resnet.children())[:-1]
resnet = nn.Sequential(*modules)
# Add projection layer
projection = nn.Linear(2048, FEATURE_DIM)
# projection = nn.Identity()
# FEATURE_DIM = 2048
resnet = resnet.to(DEVICE)
projection = projection.to(DEVICE)
resnet.eval()
projection.eval()

C:\Users\PAWANESH\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Linear(in_features=2048, out_features=512, bias=True)

In [15]:
@torch.no_grad()
def extract_feature(image_path):    
    image = Image.open(image_path).convert("RGB")    
    image = transform(image).unsqueeze(0).to(DEVICE)    
    feature = resnet(image)    
    feature = feature.view(1, -1)    
    feature = projection(feature)    
    feature = feature.squeeze(0).cpu().numpy()    
    return feature

### <div style="background-color:blue; color:white; padding:10px;"> Load all frame paths</div>

In [16]:
rgb_frames = sorted(glob.glob(os.path.join(RGB_ROOT, "*.jpg")))
flow_frames = sorted(glob.glob(os.path.join(FLOW_ROOT, "*.jpg")))
assert len(rgb_frames) == len(flow_frames), "Mismatch in RGB and Flow frames"
num_frames = len(rgb_frames)
print("Total frames:", num_frames)

Total frames: 76325


### <div style="background-color:blue; color:white; padding:10px;"> Extract RGB and Flow features </div>

In [17]:
rgb_features = np.zeros((num_frames, FEATURE_DIM), dtype=np.float32)
flow_features = np.zeros((num_frames, FEATURE_DIM), dtype=np.float32)
for i in tqdm(range(num_frames), desc="Extracting features"):    
    rgb_features[i] = extract_feature(rgb_frames[i])    
    flow_features[i] = extract_feature(flow_frames[i])
print("RGB features:", rgb_features.shape)
print("Flow features:", flow_features.shape)

Extracting features: 100%|█████████████████████████████████████████████████████| 76325/76325 [1:06:35<00:00, 19.10it/s]

RGB features: (76325, 512)
Flow features: (76325, 512)


### <div style="background-color:blue; color:white; padding:10px;"> Adaptive multimodal fusion </div>

We implement adaptive fusion layer:

$$f_t=\alpha {f_t}^{rgb} + (1-\alpha) {f_t}^{flow}$$

In [18]:
class AdaptiveFusion(nn.Module):    
    def __init__(self, dim):        
        super().__init__()        
        self.fc = nn.Linear(dim * 2, 1)       
        self.sigmoid = nn.Sigmoid()   
    def forward(self, rgb, flow):        
        x = torch.cat([rgb, flow], dim=1)        
        alpha = self.sigmoid(self.fc(x))        
        fused = alpha * rgb + (1 - alpha) * flow        
        return fused

In [19]:
# Initialize fusion:
fusion_model = AdaptiveFusion(FEATURE_DIM).to(DEVICE)
fusion_model.eval()

AdaptiveFusion(
  (fc): Linear(in_features=1024, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [20]:
# Apply fusion:
rgb_tensor = torch.from_numpy(rgb_features).to(DEVICE)
flow_tensor = torch.from_numpy(flow_features).to(DEVICE)
with torch.no_grad():   
    fused_tensor = fusion_model(rgb_tensor, flow_tensor)
fused_features = fused_tensor.cpu().numpy()
print("Fused features shape:", fused_features.shape)

Fused features shape: (76325, 512)


### <div style="background-color:blue; color:white; padding:10px;">Save fused features and labels to CSV </div>

In [21]:
features_df = pd.DataFrame(fused_features)
features_df.insert(0, "frame_id", np.arange(num_frames))
features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)
print("Saved fused features:", OUTPUT_FEATURES_CSV)

Saved fused features: ../Features/P01_05_fused_features.csv


In [22]:
# Correct alignment fix
num_frames = len(rgb_frames)
label_frames = labels.shape[0]
if label_frames < num_frames:    
    padding = np.zeros((num_frames - label_frames, labels.shape[1]), dtype=np.float32) 
    labels = np.vstack([labels, padding])
elif label_frames > num_frames:
    labels = labels[:num_frames]
print("Final labels shape:", labels.shape)
labels_df = pd.DataFrame(labels)
labels_df.insert(0, "frame_id", np.arange(num_frames))
labels_df.to_csv(OUTPUT_LABELS_CSV, index=False)
print("Saved labels:", OUTPUT_LABELS_CSV)

Final labels shape: (76325, 123)
Saved labels: ../Labels/P01_05_labels.csv
